# Project 3: Weather Analytics
**Domain:** Weather  
**Dataset Source:** `data/weather_data.csv`  

---

## Executive Overview
Time series and extreme event analysis.

---


In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.statistical_analysis import StatisticalAnalyzer
from src.visualization import Visualizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Data Quality & Preprocessing
Checklist:
✓ Dataset shape
✓ Missing values
✓ Duplicate rows
✓ Numeric outliers


In [ ]:
loader = DataLoader('../data/weather_data.csv')
df = loader.load_data()

print("==============================")
print("     RAW DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


### Domain Assumption: Missing Rainfall
Observe the RAW DATA QUALITY above. Rainfall is missing in 96.8% of the rows. If we assume 'missing' means 'no rainfall was recorded because it didn't rain', then imputing with `0.0` is the correct domain approach. We will now apply this cleaning decision.

In [ ]:
df = loader.clean_missing_values(strategy_map={'Rainfall_mm': 0.0})

print("==============================")
print("   CLEANED DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


### Outlier Analysis
**Business Question:** Are there anomalous temperature readings?


In [ ]:
sns.boxplot(data=df, y='Temperature_C')
plt.show()

**Finding:** No extreme outliers observed.  
**Meaning:** Data is well-bounded.  
**Recommendation:** Proceed with standard analysis.

### Advanced Pandas: Time Series Resampling
**Business Question:** What is the monthly average rainfall?


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
monthly = df.set_index('Date').resample('ME')['Rainfall_mm'].mean()
display(monthly.head())

### Extreme Weather Definition
**Business Question:** How often do heatwaves occur?


In [ ]:
heatwave_thresh = df['Temperature_C'].quantile(0.95)
heatwaves = df[df['Temperature_C'] > heatwave_thresh]
print(f'Defined heatwave as > {heatwave_thresh:.1f}C. Found {len(heatwaves)} instances.')

**Finding:** Heatwaves are rare but identifiable.  
**Meaning:** We have a strict 95th percentile definition.  
**Recommendation:** Issue alerts when temp exceeds 95th percentile.

### Statistical Hypothesis Testing (ANOVA) & Tukey HSD
**Business Question:** Are weather condition temperature differences statistically significant?


In [ ]:
stats = StatisticalAnalyzer(df)
res = stats.one_way_anova('WeatherCondition', 'Temperature_C')
tukey = stats.tukey_hsd_test('WeatherCondition', 'Temperature_C')
sig_pairs = tukey[tukey['is_significant']]
tukey_insight = 'Post-hoc Tukey HSD reveals significant differences between: ' + ', '.join([f"{r['group1']} vs {r['group2']}" for _, r in sig_pairs.head(3).iterrows()])
print(stats.format_hypothesis_report(
    'No difference in mean temp across conditions.', 'Mean temp differs by condition.', 'One-Way ANOVA', 'F', res['test_statistic'], res['p_value'], f'Conditions have distinct temperature profiles. {tukey_insight}', 'No significant evidence that conditions vary by temperature.', why_it_matters_reject='Useful for forecasting energy grid demand during specific weather events.'
))

## Limitations
- Dataset size is limited.
- Results are observational.
- Correlation does not imply causation.
- Some variables contain missing observations.
- External factors are not included.
